# Lesson 2: File Handling

**Week 3 · Data Engineering Course**

---

Data engineering is mostly about moving data between files, databases, and APIs. This lesson covers the three most common file operations you will do:

1. Reading and writing CSV files
2. Reading and writing JSON
3. Navigating the file system with `pathlib`

It also covers **error handling** — what to do when a file does not exist or has a problem.

In [ ]:
import csv
import json
from pathlib import Path

print('Imports ready.')

---

## 2.1 Reading a CSV File

The built-in `csv` module handles CSV files correctly. It deals with commas inside quoted fields, different line endings, and other edge cases that break a naive `.split(',')` approach.

`csv.DictReader` reads each row as a Python dictionary, with the column headers as keys.

In [ ]:
# Always use the with statement to open files
# It closes the file automatically when the block ends — even if an error occurs

csv_path = Path('../data/raw/sales_q1.csv')

with open(csv_path, 'r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    rows = list(reader)   # read all rows into a list of dicts

print(f'Rows loaded: {len(rows)}')
print(f'\nFirst row:')
for key, value in rows[0].items():
    print(f'  {key}: {repr(value)}')

In [ ]:
# repr() shows the exact content — useful for spotting hidden spaces
print('customer_name with repr:')
for row in rows[:3]:
    print(f'  {repr(row["customer_name"])}')
# You may see leading/trailing spaces like '  Bob Smith  ' — those need cleaning

In [ ]:
# Walk through the rows to inspect issues before cleaning
print('Spot check — first 5 rows:')
for row in rows[:5]:
    print(f"  order {row['order_id']:>4}  cat={row['category']!r:25}  qty={row['quantity']!r:4}  price={row['unit_price']!r}")

---

## 2.2 Writing a CSV File

`csv.DictWriter` writes a list of dictionaries to a CSV file. You give it the column names (fieldnames) and it writes the header and rows.

In [ ]:
# Write a small sample to a new file
out_path = Path('../data/clean/sample.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)  # create the folder if it does not exist

sample_rows = [
    {'order_id': '1001', 'product': 'Laptop',       'total': '1200.00'},
    {'order_id': '1002', 'product': 'Wireless Headphones', 'total': '150.00'},
]

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['order_id', 'product', 'total'])
    writer.writeheader()       # writes the column names as the first row
    writer.writerows(sample_rows)

print(f'Written to {out_path}')

# Read it back to verify
with open(out_path, 'r', newline='', encoding='utf-8') as f:
    print(f.read())

---

## 2.3 A Read-Clean-Write Pipeline

The basic pattern for every data cleaning script:
1. **Read** raw rows from the source file
2. **Clean** each row
3. **Write** the clean rows to a new file

In [ ]:
def clean_row(row):
    '''Clean one row from the sales CSV. Returns the cleaned dict.'''
    row['customer_name'] = row['customer_name'].strip()
    row['product']       = row['product'].strip()
    row['category']      = row['category'].strip().title()
    return row

raw_path   = Path('../data/raw/sales_q1.csv')
clean_path = Path('../data/clean/sales_q1_clean.csv')

# Step 1: Read
with open(raw_path, 'r', newline='', encoding='utf-8') as f:
    raw_rows = list(csv.DictReader(f))
print(f'Read {len(raw_rows)} rows')

# Step 2: Clean
cleaned = [clean_row(row) for row in raw_rows]

# Step 3: Write
clean_path.parent.mkdir(parents=True, exist_ok=True)
with open(clean_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(cleaned[0].keys()))
    writer.writeheader()
    writer.writerows(cleaned)

print(f'Wrote {len(cleaned)} rows to {clean_path}')

---

## 2.4 JSON

JSON is the standard format for APIs and configuration files. Python's `json` module converts between JSON files/strings and Python dicts and lists.

In [ ]:
# A Python dict -> JSON file
config = {
    'database':   'sales_db',
    'host':       'localhost',
    'port':       5432,
    'tables':     ['orders', 'products', 'customers']
}

config_path = Path('../data/clean/config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

print('Saved config:')
print(config_path.read_text())

In [ ]:
# JSON file -> Python dict
with open(config_path, 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print(type(loaded))              # <class 'dict'>
print(loaded['database'])        # sales_db
print(loaded['tables'][0])       # orders

---

## 2.5 pathlib — Working with File Paths

`pathlib.Path` is the modern way to work with file and folder paths. It is cleaner and more readable than building paths as strings, and it works on Windows, Mac, and Linux without changes.

In [ ]:
# Build paths with /
base = Path('../data')
raw_dir   = base / 'raw'
clean_dir = base / 'clean'
q1_file   = raw_dir / 'sales_q1.csv'

print(q1_file)                   # ../data/raw/sales_q1.csv
print(q1_file.name)              # sales_q1.csv
print(q1_file.stem)              # sales_q1
print(q1_file.suffix)            # .csv
print(q1_file.parent)            # ../data/raw

In [ ]:
# Check if a path exists
print(raw_dir.exists())    # True
print(raw_dir.is_dir())    # True
print(q1_file.is_file())   # True

# Find all CSV files in a folder
csv_files = sorted(raw_dir.glob('*.csv'))
for f in csv_files:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name}  ({size_kb:.1f} KB)')

In [ ]:
# Create a folder (does nothing if it already exists)
output_folder = Path('../data/clean')
output_folder.mkdir(parents=True, exist_ok=True)
print(f'Folder ready: {output_folder}')

---

## 2.6 Handling Errors

Files are where things go wrong: the path is wrong, the folder does not exist, permissions are missing, the file is locked. `try/except` lets you handle these situations without crashing.

In [ ]:
# Without error handling — crashes if the file is missing
# with open(Path('nonexistent.csv'), 'r') as f:
#     data = f.read()   # FileNotFoundError

# With error handling
def read_csv_safe(path):
    '''Read a CSV file. Returns an empty list if the file does not exist.'''
    try:
        with open(path, 'r', newline='', encoding='utf-8') as f:
            return list(csv.DictReader(f))
    except FileNotFoundError:
        print(f'File not found: {path}')
        return []
    except PermissionError:
        print(f'No permission to read: {path}')
        return []

result = read_csv_safe(Path('nonexistent.csv'))
print(f'Result: {result}')   # []

result = read_csv_safe(Path('../data/raw/sales_q1.csv'))
print(f'Loaded {len(result)} rows')

In [ ]:
# Process multiple files — skip the ones that fail
files_to_load = [
    Path('../data/raw/sales_q1.csv'),
    Path('../data/raw/missing_file.csv'),   # does not exist
    Path('../data/raw/sales_q2.csv'),
]

all_rows = []
for path in files_to_load:
    rows = read_csv_safe(path)
    if rows:
        all_rows.extend(rows)
        print(f'Loaded {len(rows)} rows from {path.name}')

print(f'\nTotal rows: {len(all_rows)}')

In [ ]:
# Clean up the files created in this lesson
for p in [out_path, clean_path, config_path]:
    if p.exists():
        p.unlink()
        print(f'Deleted {p.name}')

---

## Key Takeaways

1. Always open files with `with open(...) as f:` — it closes the file automatically.
2. Always specify `encoding='utf-8'` when opening text files.
3. `csv.DictReader` gives you each row as a dictionary — one key per column header.
4. `csv.DictWriter` writes a list of dictionaries to a CSV. Always call `.writeheader()` first.
5. `json.load(f)` reads a JSON file into a Python dict. `json.dump(data, f)` writes a dict to a JSON file.
6. Use `pathlib.Path` for all file paths. Join folders with `/`. Use `.glob('*.csv')` to find files.
7. Wrap file operations in `try/except FileNotFoundError` so one missing file does not stop the whole pipeline.